# Adaptive Inference Router - Google Colab Demo

Este notebook demuestra el uso del **Adaptive Inference Router** basado en el framework de Intention Collapse.

## ¿Qué hace el router?
- Mide la **intention entropy H_int(I)** antes de generar respuestas
- **Baja entropía** → respuesta directa (barato, rápido)
- **Alta entropía** → Chain-of-Thought (más tokens, mejor razonamiento)

## 🚀 Uso:
1. **Ejecuta la celda de instalación** (celda siguiente) - todo automático
2. **Carga el modelo** (GPT-2 para demo rápido)
3. **Ejecuta las 4 preguntas de prueba** y observa las decisiones del router
4. **Visualiza resultados** con gráficos de entropía y token usage

## ⏱️ Tiempo total: ~3-5 minutos

**¡Empecemos!** 👇

## 1. Setup Completo: Instalación y Verificación

**Esta celda ejecuta todo lo necesario:**
- Clona el repositorio desde GitHub
- Instala el paquete y dependencias
- Configura el path de Python
- Verifica que todo funcione

**⏱️ Tiempo estimado: ~30 segundos**

Simplemente ejecuta la celda siguiente y espera el mensaje: **"🎉 INSTALACIÓN COMPLETA"**

In [ ]:
import sys
import subprocess

print("=" * 70)
print("INSTALACIÓN AUTOMÁTICA - Adaptive Inference Router")
print("=" * 70)

# 1. Limpiar instalaciones previas
print("\n[1/4] Limpiando instalación previa...")
subprocess.run(["rm", "-rf", "/content/intention-collapse-experiments"], 
               capture_output=True, check=False)

# 2. Clonar repositorio
print("[2/4] Clonando repositorio desde GitHub...")
result = subprocess.run(
    ["git", "clone", "-q", "-b", "v2-router-experiments",
     "https://github.com/patriciomvera/intention-collapse-experiments.git",
     "/content/intention-collapse-experiments"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(f"❌ Error clonando: {result.stderr}")
    raise RuntimeError("Git clone failed")

print("   ✅ Repositorio clonado")

# 3. Instalar paquete
print("[3/4] Instalando paquete intention-collapse...")
result = subprocess.run(
    ["pip", "install", "-q", "-e", "/content/intention-collapse-experiments/"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(f"⚠️  Pip install tuvo warnings (pero puede funcionar):")
    if result.stderr:
        # Mostrar solo las últimas líneas del error si es muy largo
        error_lines = result.stderr.split('\n')
        print('\n'.join(error_lines[-10:]))
else:
    print("   ✅ Paquete instalado")

# 4. Configurar Python path (crítico para Colab)
print("[4/4] Configurando Python path...")
if '/content/intention-collapse-experiments' not in sys.path:
    sys.path.insert(0, '/content/intention-collapse-experiments')
    print("   ✅ Path configurado")

# 5. Verificar instalación
print("\n" + "=" * 70)
print("VERIFICACIÓN DE INSTALACIÓN")
print("=" * 70)

try:
    from src.router import AdaptiveInferenceRouter, RouteDecision
    from src.metrics import compute_intention_entropy
    from src.controls import self_consistency_baseline
    
    print("✅ Todos los imports exitosos!")
    print(f"✅ RouteDecision disponible: {[r.value for r in RouteDecision]}")
    print(f"✅ AdaptiveInferenceRouter: {AdaptiveInferenceRouter.__name__}")
    
    print("\n" + "=" * 70)
    print("🎉 INSTALACIÓN COMPLETA - Listo para ejecutar!")
    print("=" * 70)
    print("\n➡️  Continúa ejecutando las siguientes celdas del notebook\n")
    
except ImportError as e:
    print(f"\n❌ Error en imports: {e}")
    print("\n🔧 Información de debug:")
    print(f"   sys.path[0]: {sys.path[0]}")
    import os
    if os.path.exists('/content/intention-collapse-experiments/src'):
        print(f"   src/ existe con: {os.listdir('/content/intention-collapse-experiments/src')[:5]}")
    raise

In [ ]:
# Verificar que los imports funcionen
# NOTA: Si esta celda falla, asegúrate de haber ejecutado la celda de instalación arriba

from src.router import AdaptiveInferenceRouter, RouteDecision
from src.metrics import compute_intention_entropy
from src.controls import self_consistency_baseline

print("✅ All imports successful!")
print(f"✅ RouteDecision options: {list(RouteDecision)}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Cargar modelo pequeño para demo
# Puedes usar: "gpt2", "facebook/opt-350m", "EleutherAI/pythia-410m"
MODEL_NAME = "gpt2"

print(f"Loading model: {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f"[OK] Model loaded on: {model.device}")
print(f"[OK] Using {'GPU' if torch.cuda.is_available() else 'CPU'}")

## 2. Cargar Modelo

Usaremos un modelo pequeño para el demo (puedes cambiarlo a modelos más grandes si tienes GPU)

In [ ]:
# Crear router con umbral de entropía
router = AdaptiveInferenceRouter(
    model=model,
    tokenizer=tokenizer,
    entropy_threshold=3.0,  # Umbral: H_int > 3.0 → CoT
    verbose=True
)

print("[OK] Router initialized")
print(f"    Entropy threshold: {router.entropy_threshold}")
print(f"    Decision logic:")
print(f"      - H_int <= {router.entropy_threshold} → DIRECT answer")
print(f"      - H_int > {router.entropy_threshold} → CoT reasoning")

## 3. Inicializar Router

El router usa intention entropy para decidir automáticamente la estrategia de inferencia.

In [ ]:
# Preguntas de prueba con diferentes niveles de dificultad
test_questions = [
    {
        "question": "What is 2 + 2?",
        "expected_route": "DIRECT",
        "reason": "Simple arithmetic - low entropy expected"
    },
    {
        "question": "If a train leaves Chicago at 3pm going 60mph and another leaves New York at 4pm going 80mph, when do they meet?",
        "expected_route": "COT",
        "reason": "Complex word problem - high entropy expected"
    },
    {
        "question": "What is the capital of France?",
        "expected_route": "DIRECT",
        "reason": "Factual knowledge - low entropy expected"
    },
    {
        "question": "A baker has 12 cookies. He gives 1/3 to his friend and eats 2. How many remain?",
        "expected_route": "COT",
        "reason": "Multi-step reasoning - high entropy expected"
    },
]

print(f"Testing router with {len(test_questions)} questions...\n")
print("="*80)

## 4. Experimento: 4 Preguntas de Prueba

Probamos el router con preguntas de diferentes niveles de complejidad.

In [ ]:
results = []

for i, test_case in enumerate(test_questions, 1):
    print(f"\n[Question {i}/{len(test_questions)}]")
    print(f"Q: {test_case['question']}")
    print(f"Expected route: {test_case['expected_route']} ({test_case['reason']})")
    print("-" * 80)
    
    # Ejecutar router
    result = router.route_and_generate(
        question=test_case['question'],
        max_new_tokens=100
    )
    
    results.append({
        'question': test_case['question'],
        'expected_route': test_case['expected_route'],
        'actual_route': result.route_taken.value.upper(),
        'entropy': result.intention_entropy,
        'answer': result.extracted_answer,
        'tokens': result.output_tokens,
        'cost_estimate': result.cost_estimate
    })
    
    # Mostrar resultado
    print(f"\n[RESULT]")
    print(f"  Route taken: {result.route_taken.value.upper()}")
    print(f"  Intention entropy: {result.intention_entropy:.3f}")
    print(f"  Answer: {result.extracted_answer}")
    print(f"  Tokens: {result.output_tokens}")
    print(f"  Cost estimate: ${result.cost_estimate:.6f}")
    
    match = "✓" if result.route_taken.value.upper() == test_case['expected_route'] else "✗"
    print(f"  Match expected: {match}")
    print("="*80)

print("\n[OK] All experiments completed!")

## 5. Ejecutar Router

Ahora ejecutamos el router en cada pregunta y observamos sus decisiones.

In [ ]:
import pandas as pd

# Crear DataFrame con resultados
df = pd.DataFrame(results)

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print("\nRouting Accuracy:")
correct = sum(df['expected_route'] == df['actual_route'])
total = len(df)
print(f"  {correct}/{total} correct ({100*correct/total:.1f}%)")

print("\nRoute Distribution:")
print(df['actual_route'].value_counts())

print("\nEntropy Statistics:")
print(f"  Mean: {df['entropy'].mean():.3f}")
print(f"  Std:  {df['entropy'].std():.3f}")
print(f"  Min:  {df['entropy'].min():.3f}")
print(f"  Max:  {df['entropy'].max():.3f}")

print("\nCost Analysis:")
total_cost = df['cost_estimate'].sum()
print(f"  Total cost: ${total_cost:.6f}")
print(f"  Avg per query: ${total_cost/len(df):.6f}")

print("\nDetailed Results:")
display(df[['question', 'actual_route', 'entropy', 'tokens', 'cost_estimate']])

## 6. Resumen de Resultados

Análisis cuantitativo del comportamiento del router.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Entropy por pregunta
colors = ['green' if r == 'DIRECT' else 'orange' for r in df['actual_route']]
ax1.bar(range(len(df)), df['entropy'], color=colors, alpha=0.7)
ax1.axhline(y=router.entropy_threshold, color='red', linestyle='--', 
            label=f'Threshold ({router.entropy_threshold})')
ax1.set_xlabel('Question Index')
ax1.set_ylabel('Intention Entropy H_int(I)')
ax1.set_title('Intention Entropy per Question')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Token usage por ruta
route_tokens = df.groupby('actual_route')['tokens'].mean()
ax2.bar(route_tokens.index, route_tokens.values, 
        color=['green', 'orange'], alpha=0.7)
ax2.set_xlabel('Route')
ax2.set_ylabel('Average Tokens')
ax2.set_title('Token Usage by Route')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("[OK] Visualization complete!")

## 7. Visualizaciones

Gráficos de entropía y uso de tokens.

In [ ]:
# Prueba con tu propia pregunta
custom_question = "What is 15 multiplied by 23?"  # Cambia esto

print(f"Testing custom question: {custom_question}\n")

result = router.route_and_generate(
    question=custom_question,
    max_new_tokens=150
)

print(f"Route: {result.route_taken.value.upper()}")
print(f"Entropy: {result.intention_entropy:.3f}")
print(f"Answer: {result.extracted_answer}")
print(f"Full response: {result.generated_text}")

## 8. Prueba con tu Propia Pregunta

Modifica `custom_question` y prueba el router con tus propias preguntas.